In [ ]:

from core import *

In [ ]:
KEY = "ASPECT" #"ALGALTOX"
d,_ = VersionedYaml(Path(WORKSPACE) / KEY).read("tags")
d

In [ ]:
d['tags']

In [ ]:
def flatten_tags(obj, parent_key="", sep="."):
    flat = {}

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)
            flat.update(flatten_tags(v, new_key, sep=sep))

    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            # Use [i] for list index keys
            new_key = f"{parent_key}[{i}]" if parent_key else f"[{i}]"
            flat.update(flatten_tags(v, new_key, sep=sep))

    else:
        # Leaf value
        flat[parent_key] = obj

    return flat

In [ ]:
flat_tags = flatten_tags(d["tags"])

# keys containing either 'explanation' OR 'confidence'
filtered_both = {
    k: v for k, v in flat_tags.items()
    if not("explanation" in k.lower() or "confidence" in k.lower())
}

print(filtered_both)

In [ ]:
KEY = "LuxPest_DB" #"40CFR302" #"ALGALTOX"

class SMILES_Data(BaseModel):  # Phase of the molecule
    SMILES_columns: List[str] = Field(default_factory=list, description="Columns in csv file that contain SMILES")

class SMILESandCSV_Data(SMILES_Data):
    CSV_file_name: str = Field(description="Name of the CSV file.")


class EssentialColumnAnalysis(BaseModel):
    file_and_columns: List[SMILESandCSV_Data]

d,_ = VersionedYaml(Path(WORKSPACE) / KEY).read("ess_columns"); print(d)

typed_d = EssentialColumnAnalysis.model_validate(d)   # pydantic v2
typed_d


In [ ]:
from inspect import TPFLAGS_IS_ABSTRACT
import pandas as pd

from processing import *
from utils import *

class SmilesRecordsIterable:
    def __init__(self, workspace: str, dataset_key: str, typed_cfg: EssentialColumnAnalysis):
        self.workspace = workspace
        self.dataset_key = dataset_key
        self.typed_cfg = typed_cfg

    def __iter__(self):
        for file_cfg in self.typed_cfg.file_and_columns:
            file_key = file_cfg.CSV_file_name
            path = Path(self.workspace) / self.dataset_key / file_key
            df = pd.read_csv(path)

            tags_, _ = VersionedYaml(Path(self.workspace) / self.dataset_key).read("tags")
            flat_tags = flatten_tags(tags_["tags"])
            filtered_both = {
                k: v for k, v in flat_tags.items()
                if not("explanation" in k.lower() or "confidence" in k.lower())
            }  

            for column_key in file_cfg.SMILES_columns:
                for raw_smiles in df[column_key]:
                    for substructure_key, sub_smiles in enumerate(split_smiles(raw_smiles)):
                        yield {
                            "SMILES": sub_smiles,
                            "dataset_key": self.dataset_key,
                            "file_key": file_key,
                            "column_key": column_key,
                            "substructure_key": substructure_key,
                            "tags": filtered_both
                        }


records = SmilesRecordsIterable(WORKSPACE, KEY, typed_d)
it = iter(records)
[next(it) for i in range(10)]

In [ ]:
# Run full SMILES processing for all keys
all_keys = sorted(
    p.name
    for p in Path(WORKSPACE).iterdir()
    if p.is_dir() and not p.name.startswith(".")
)

all_keys

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

ALL_DATA = {}

for dataset_key in tqdm(all_keys[:], desc="Datasets"):
    pair_ = VersionedYaml(Path(WORKSPACE) / dataset_key).read("tags")
    if pair_ is not None:
        tags_, _ = pair_
        flat_tags = flatten_tags(tags_["tags"])
        tags = {
            k: v for k, v in flat_tags.items()
            if not ("explanation" in k.lower() or "confidence" in k.lower())
        }

        unique_smiles_path = Path(WORKSPACE_DATA) / dataset_key / "unique_smiles2.csv"
        if os.path.exists(unique_smiles_path):
            df = pd.read_csv(unique_smiles_path, on_bad_lines="skip")
            smiles_col = "norm_SMILES"

            for smiles in df[smiles_col].dropna().astype(str):
                if '*' not in smiles:
                    if smiles not in ALL_DATA:
                        ALL_DATA[smiles] = {
                            "SMILES": smiles,
                            "dataset_key": [dataset_key],
                            **tags,
                        }
                    else:
                        if dataset_key not in ALL_DATA[smiles]["dataset_key"]:
                            ALL_DATA[smiles]["dataset_key"].append(dataset_key)

                        for tag_key, tag_value in tags.items():
                            ALL_DATA[smiles][tag_key] = ALL_DATA[smiles].get(tag_key, False) or tag_value

len(ALL_DATA)

In [ ]:
df = pd.DataFrame(ALL_DATA.values())
os.makedirs(Path(WORKSPACE_SUMMARY), exist_ok=True)
df.to_csv(Path(WORKSPACE_SUMMARY)/'dataset-of-datasets-unique-smiles.csv')

In [ ]:
df

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

ALL_DATA = {}

for dataset_key in tqdm(all_keys[:], desc="Datasets"):
    pair_ = VersionedYaml(Path(WORKSPACE) / dataset_key).read("tags")
    if pair_ is not None:
        tags_, _ = pair_
        flat_tags = flatten_tags(tags_["tags"])
        tags = {
            k: v for k, v in flat_tags.items()
            if not ("explanation" in k.lower() or "confidence" in k.lower())
        }

        unique_smiles_path = Path(WORKSPACE_DATA) / dataset_key / "unique_smiles2.csv"
        if os.path.exists(unique_smiles_path):
            df = pd.read_csv(unique_smiles_path, on_bad_lines="skip")
            smiles_col = "norm_SMILES"

            for smiles in df[smiles_col].dropna().astype(str):
                if '*' in smiles:
                    if smiles not in ALL_DATA:
                        ALL_DATA[smiles] = {
                            "SMILES": smiles,
                            "dataset_key": [dataset_key],
                            **tags,
                        }
                    else:
                        if dataset_key not in ALL_DATA[smiles]["dataset_key"]:
                            ALL_DATA[smiles]["dataset_key"].append(dataset_key)

                        for tag_key, tag_value in tags.items():
                            ALL_DATA[smiles][tag_key] = ALL_DATA[smiles].get(tag_key, False) or tag_value

len(ALL_DATA)

In [ ]:
df = pd.DataFrame(ALL_DATA.values())
os.makedirs(Path(WORKSPACE_SUMMARY), exist_ok=True)
df.to_csv(Path(WORKSPACE_SUMMARY)/'dataset-of-datasets-templates.csv')

In [ ]:
df